In [ ]:
import os
import chardet
import pandas as pd
import librosa
import glob
from pathlib import Path
import random
import numpy as np
from math import sqrt
import sys
from transformers import T5Tokenizer, T5EncoderModel
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from tqdm import trange
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,Dataset
from torchaudio import transforms
import torch.nn.functional as F
import soundfile as sf
import pywt
import scipy.signal as signal
import noisereduce as nr
from typing import List, Tuple, Dict
from IPython.display import Audio, display
from torch.utils.data.distributed import DistributedSampler

In [ ]:
df_birds = pd.read_csv('df_birds.csv')
df_birds = df_birds.dropna(subset=['relative_path']).reset_index(drop=True)
df_birds.head()

In [ ]:
!git clone https://github.com/lmnt-com/diffwave.git
!cd diffwave

In [ ]:
%cd /content/diffwave
!ls -l
!pip install .

## Preprocessing for enhancement

In [ ]:
def est_snr(signal: np.ndarray, noise: np.ndarray) -> float:
    p_sig = np.mean(signal**2)
    p_n   = np.mean(noise**2) + 1e-8
    return 10 * np.log10(p_sig / p_n + 1e-8)


def seg_snr(signal: np.ndarray, denoised: np.ndarray,
                  frame_len: float = 0.02, sr: int = 16000) -> float:
    L = int(frame_len * sr)
    snr_list = []
    for i in range(0, len(signal), L):
        seg = signal[i:i+L]
        den = denoised[i:i+L]
        if np.mean(seg**2) < 1e-6:
            continue
        numerator   = np.sum(seg**2)
        denominator = np.sum((seg - den)**2) + 1e-8
        snr_frame   = 10 * np.log10(numerator / denominator)
        snr_list.append(snr_frame)
    return np.mean(snr_list) if snr_list else 0.0


def band_weight(band_signal: np.ndarray, original: np.ndarray,
                          sr: int, low_freq: float, high_freq: float) -> float:
    band_energy  = np.mean(band_signal**2)
    total_energy = np.mean(original**2) + 1e-8
    energy_ratio = band_energy / total_energy

    freqs, psd = sps.welch(band_signal, sr, nperseg=1024)
    freq_mask = (freqs >= low_freq) & (freqs <= high_freq)
    if np.any(freq_mask):
        peak_power = np.max(psd[freq_mask])
        avg_power  = np.mean(psd[freq_mask]) + 1e-8
        spectral_pro = peak_power / avg_power
    else:
        spectral_pro = 1.0

    envelope = np.abs(sps.hilbert(band_signal))
    act = np.std(envelope) / (np.mean(envelope) + 1e-8)

    weight = (
        energy_ratio * 0.4
        + np.log10(spectral_pro + 1) * 0.3
        + np.tanh(act) * 0.3
    )
    return max(0.1, min(1.0, weight))


def noise_reduce(snr_db: float, base_strength: float) -> float:
    if snr_db > 10:
        return base_strength * 0.6
    elif snr_db > 5:
        return base_strength
    elif snr_db > 0:
        return base_strength * 1.3
    else:
        return min(0.8, base_strength * 1.6)


def select_noise(residual: np.ndarray, sr: int,
                          frame_duration: float = 0.5) -> np.ndarray:
    frame_len = int(sr * frame_duration)
    n_samples = min(5, len(residual) // frame_len)
    positions = np.linspace(0, len(residual) - frame_len, n_samples, dtype=int)

    segments = []
    features = []
    for pos in positions:
        segment = residual[pos:pos+frame_len]
        segments.append(segment)

        energy   = np.mean(segment**2)
        variance = np.var(segment)
        zcr      = np.mean(np.abs(np.diff(np.sign(segment)))) / 2

        features.append([energy, variance, zcr])

    features = np.array(features)
    features_norm = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)
    scores = np.sum(features_norm, axis=1)
    best_idx = np.argmin(scores)
    return segments[best_idx]  


def compress(signal: np.ndarray, threshold: float, ratio: float) -> np.ndarray:
    peak = np.max(np.abs(signal)) + 1e-8
    signal = signal / peak

    def soft_comp(x, thresh, ratio):
        abs_x = np.abs(x)
        mask = abs_x > thresh
        compressed = np.copy(x)
        over = abs_x[mask] - thresh
        compressed[mask] = np.sign(x[mask]) * (thresh + over / (1 + over * ratio))
        return compressed

    return soft_comp(signal, threshold, ratio)


def MSAD(
    waveform: np.ndarray,
    sr: int,
    freq_bands: List[Tuple[float, float]] = None,
    adaptive_params: bool = True,
    base_prop: float = 0.4,
    dyn_threshold: float = 0.1,
    dyn_ratio: float = 2.0
) -> Tuple[np.ndarray, List[float], float, float]:

    if freq_bands is None:
        freq_bands = [
            (1500, 3000),  
            (2500, 5000),  
            (4000, 8000),  
            (6000, 12000),  
        ]

    # Estimating original bird calls and noise
    sos_rough = sps.butter(4, [2000/(sr/2), 8000/(sr/2)], btype='bandpass',output='sos')
    rough_signal = sps.sosfiltfilt(sos_rough, waveform)
    rough_noise= waveform - rough_signal

    orig_snr = seg_snr(rough_signal, rough_noise, frame_len=0.02, sr=sr)
    print(f"original: {orig_snr:+.2f} dB")

    # multi-scale frequency bands
    band_signals = []
    band_weights = []
    for i, (low_freq, high_freq) in enumerate(freq_bands):
        sos = sps.butter(4,[low_freq/(sr/2), high_freq/(sr/2)],btype='bandpass',output='sos')
        band_signal = sps.sosfiltfilt(sos, waveform)
        weight = band_weight(band_signal, waveform, sr, low_freq, high_freq)  
        band_signals.append(band_signal)
        band_weights.append(weight)
        print(f"band {i+1}: {low_freq}-{high_freq} hz, weight = {weight:.3f}")

    # weighted bird calls signal
    weighted_signal = np.zeros_like(waveform)
    total_weight = sum(band_weights)
    for band_signal, w in zip(band_signals, band_weights):
        weighted_signal += band_signal * (w / total_weight)

    residual = waveform - weighted_signal

    # spectral subtraction strength
    if adaptive_params:
        cur_snr = seg_snr(weighted_signal, residual, frame_len=0.02, sr=sr)
        adaptive_prop = noise_reduce(cur_snr, base_prop)  
        print(f"estimated SNR: {cur_snr:+.2f} db, decrease = {adaptive_prop:.3f}")  
    else:
        adaptive_prop = base_prop  

    # noise reference
    noise_clip = select_noise(residual, sr)

    # denoising
    residual_denoised = nr.reduce_noise(
        y = residual,
        y_noise = noise_clip,
        sr = sr,
        prop_decrease = adaptive_prop,
        stationary = False
    )

    enhanced_snr = seg_snr(weighted_signal, residual_denoised, frame_len=0.02, sr=sr)
    print(f"enhanced: {enhanced_snr:+.2f} db   ({enhanced_snr - orig_snr:+.2f} db)")

    # final signal
    clean = weighted_signal + residual_denoised

    # normalization and compression
    peak = np.max(np.abs(clean)) + 1e-8
    clean = clean / peak

    mask = np.abs(clean) > dyn_threshold
    clean[mask] = np.sign(clean[mask]) * (dyn_threshold + (np.abs(clean[mask]) - dyn_threshold) / dyn_ratio)

    return clean, band_weights, orig_snr, enhanced_snr


if __name__ == "__main__":   
    freq_bands = [(500, 2000),(1500, 3000),(2500, 8000),(7000, 10000)]
    wav, sr = torchaudio.load("/content/data/1331/94492_1.wav")
    wav = wav[0].numpy()

    enhanced, band_weights, orig_snr, enhanced_snr = MSAD(
        waveform           = wav,
        sr                 = sr,
        freq_bands         = freq_bands,  
        adaptive_params    = True,
        base_prop          = 0.4,  
        dyn_threshold      = 0.1,
        dyn_ratio          = 2.0
    )

    plt.figure(figsize=(15, 8))
    plt.subplot(3, 1, 1)
    plt.plot(wav);   plt.title(f"Original Audio (Segmental SNR: {orig_snr:.1f} dB)")
    plt.subplot(3, 1, 2)
    plt.plot(enhanced)
    plt.title(f"Enhanced Audio (Segmental SNR: {enhanced_snr:.1f} dB)")
    plt.tight_layout();
    plt.show()

    display(Audio(wav, rate=sr))
    display(Audio(enhanced, rate=sr))

In [ ]:
def enhance_data(
    input_folder: str,
    output_folder: str,
    freq_bands: list = None,
    audio_extensions: tuple = ('.wav', '.mp3', '.flac', '.m4a'),
    base_prop: float = 0.4,
    overwrite: bool = False
):
    
    os.makedirs(output_folder, exist_ok=True)
    
    if freq_bands is None:
        freq_bands = [(500, 2000), (1500, 3000), (2500, 8000), (7000, 10000)]
    
    audio_files = []
    for ext in audio_extensions:
        pattern = os.path.join(input_folder, f"**/*{ext}")
        audio_files.extend(glob.glob(pattern, recursive=True))
    
    if not audio_files:
        print(f"no files")
        return
       
    processed_count = 0
    skipped_count = 0
    error_count = 0
    snr_improvements = []
    
    for audio_path in tqdm(audio_files, desc="Processing Audio"):
        try:
            rel_path = os.path.relpath(audio_path, input_folder)
            output_path = os.path.join(output_folder, rel_path)
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            if os.path.exists(output_path) and not overwrite:
                skipped_count += 1
                continue
            
            wav, sr = torchaudio.load(audio_path)
            
            if wav.shape[0] > 1:
                wav = wav[0]
            wav = wav.numpy()
            
            enhanced, weights, orig_snr, enhanced_snr = MSAD(
                waveform=wav,
                sr=sr,
                freq_bands=freq_bands,
                adaptive_params=True,
                base_prop=base_prop,
                verbose=False 
            )
            
            enhanced_tensor = torch.from_numpy(enhanced).unsqueeze(0)
            torchaudio.save(output_path, enhanced_tensor, sr)
            
            processed_count += 1
            snr_improvements.append(enhanced_snr - orig_snr)
            
           
        except Exception as e:
            print(f"{rel_path} has error: {str(e)}")
            error_count += 1
            continue   

if __name__ == "__main__":
    raw = "/data/"     
    output = "/data_new/" 
    freq_bands = [(500, 2000),(1500, 3000), (2500, 8000),(7000, 10000)]
    
    enhance_data(
        input_folder=raw,
        output_folder=output,
        freq_bands=freq_bands,
        base_prop=0.4,
        overwrite=False  
    )

## get specs and text

In [ ]:
# specs by diffwave preprocess
!python -m diffwave.preprocess /content/drive/MyDrive/data_new/

In [ ]:
# text embedding by T5
df_birds["text"] = df_birds["text"].fillna("").astype(str)

tokenizer = T5Tokenizer.from_pretrained("t5-small")
encoder = T5EncoderModel.from_pretrained("t5-small")
encoder.eval()

def encode_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = encoder(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze(0).numpy()  

emb_list = []
for i, row in df_birds.iterrows():
    text_str = row["text"]
    emb = encode_text(text_str)
    emb_list.append(emb)

# save    
emb_array = np.array(emb_list)
np.save("text_embeddings.npy", emb_array)

In [ ]:
class AudioUtil:
    @staticmethod
    def open(audio_file, sr=22050):
        sig, sr = librosa.load(audio_file, sr=sr, mono=True)
        sig = torch.tensor(sig).unsqueeze(0)
        return sig

class SoundDataset(Dataset):
    def __init__(self, df, embedding_file):
        self.df = df.reset_index(drop=True) 
        all_embeddings = np.load(embedding_file)
        self.original_indices = df.index.tolist()
        self.embeddings = all_embeddings[self.original_indices]
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        wave_path = self.df.loc[idx, 'relative_path']
        class_id = self.df.loc[idx, 'class_id']
        aud = AudioUtil.open(wave_path)
        spec_path = wave_path.replace("/data/", "/data_new/") \
                            .replace(".wav", ".wav.spec.npy")
        spectrogram = np.load(spec_path)

        signal = aud[0].numpy().astype(np.float32)
        spectrogram = spectrogram.astype(np.float32)
        
        text_embedding = self.embeddings[idx]
        text_embedding = torch.from_numpy(text_embedding).float()

        return signal, spectrogram, class_id, text_embedding 



## model and training

In [ ]:
#%%writefile params.py

class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self
    def override(self, attrs):
        if isinstance(attrs, dict):
            self.__dict__.update(**attrs)
        elif isinstance(attrs, (list, tuple, set)):
            for attr in attrs:
                self.override(attr)
        elif attrs is not None:
            raise NotImplementedError
        return self

params = AttrDict(
    # training params
    batch_size=16,
    learning_rate=1e-4,
    max_grad_norm=None,
    # data params
    sample_rate=22050,
    n_mels=80,
    n_fft=1024,
    hop_samples=256,
    crop_mel_frames=62,
    num_labels=20,
    # model params
    residual_layers=30,
    residual_channels=64,
    dilation_cycle_length=10,
    unconditional=False,
    noise_schedule=np.linspace(1e-4, 0.05, 50).tolist(),
    inference_noise_schedule=[0.0001, 0.001, 0.01, 0.05, 0.2, 0.5],
    audio_len=22050*5,
)

In [ ]:
#%%writefile diffwave.py

class HierarchicalConditionFusion(nn.Module):
    def __init__(self, n_mels, num_heads=4):
        super().__init__()
        self.condition_weights = nn.Parameter(torch.ones(3))
        self.use_hierarchical = True

    def forward(self, c_spec, c_label=None, c_text=None):
        if not self.use_hierarchical:
            result = c_spec
            if c_label is not None:
                c_label_expanded = c_label.expand(-1, -1, c_spec.shape[-1])
                result = result + 0.1 * c_label_expanded
            if c_text is not None:
                c_text_expanded = c_text.expand(-1, -1, c_spec.shape[-1])
                result = result + 0.1 * c_text_expanded
            return result

        # Adaptive Weight Fusion
        weights = F.softmax(self.condition_weights, dim=0)

        result = weights[0] * c_spec
        if c_label is not None:
            c_label_expanded = c_label.expand(-1, -1, c_spec.shape[-1])
            result = result + weights[1] * c_label_expanded
        if c_text is not None:
            c_text_expanded = c_text.expand(-1, -1, c_spec.shape[-1])
            result = result + weights[2] * c_text_expanded

        return result

Linear = nn.Linear
def Conv1d(*args, **kwargs):
    layer = nn.Conv1d(*args, **kwargs)
    nn.init.kaiming_normal_(layer.weight)
    return layer

@torch.jit.script
def silu(x):
    return x * torch.sigmoid(x)

class DiffusionEmbedding(nn.Module):
    def __init__(self, max_steps):
        super().__init__()
        self.register_buffer('embedding', self._build_embedding(max_steps), persistent=False)
        self.projection1 = Linear(128, 512)
        self.projection2 = Linear(512, 512)
    def forward(self, diffusion_step):
        if diffusion_step.dtype in [torch.int32, torch.int64]:
            x = self.embedding[diffusion_step]
        else:
            x = self._lerp_embedding(diffusion_step)
        x = F.silu(self.projection1(x))
        x = F.silu(self.projection2(x))
        return x
    def _lerp_embedding(self, t):
        low_idx = torch.floor(t).long()
        high_idx = torch.ceil(t).long()
        low = self.embedding[low_idx]
        high = self.embedding[high_idx]
        return low + (high - low) * (t - low_idx)
    def _build_embedding(self, max_steps):
        steps = torch.arange(max_steps).unsqueeze(1)
        dims = torch.arange(64).unsqueeze(0)
        table = steps * 10.0**(dims * 4.0 / 63.0)
        table = torch.cat([torch.sin(table), torch.cos(table)], dim=1)
        return table

class SpectrogramUpsampler(nn.Module):
    def __init__(self, n_mels):
        super().__init__()
        self.conv1 = nn.ConvTranspose2d(1, 1, [3, 32], stride=[1, 16], padding=[1, 8])
        self.conv2 = nn.ConvTranspose2d(1, 1, [3, 32], stride=[1, 16], padding=[1, 8])
    def forward(self, x):
        x = torch.unsqueeze(x, 1)
        x = self.conv1(x)
        x = F.leaky_relu(x, 0.4)
        x = self.conv2(x)
        x = F.leaky_relu(x, 0.4)
        x = torch.squeeze(x, 1)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, n_mels, residual_channels, dilation, uncond=False):
        super().__init__()
        self.dilated_conv = Conv1d(residual_channels, 2 * residual_channels, 3, padding=dilation, dilation=dilation)
        self.diffusion_projection = Linear(512, residual_channels)
        if not uncond:
            self.conditioner_projection = Conv1d(n_mels, 2 * residual_channels, 1)
        else:
            self.conditioner_projection = None
        self.output_projection = Conv1d(residual_channels, 2 * residual_channels, 1)
    def forward(self, x, diffusion_step, conditioner=None):
        diffusion_step = self.diffusion_projection(diffusion_step).unsqueeze(-1)
        y = x + diffusion_step
        if self.conditioner_projection is not None and conditioner is not None:
            cond = self.conditioner_projection(conditioner)
            y = self.dilated_conv(y) + cond
        else:
            y = self.dilated_conv(y)
        gate, filter = torch.chunk(y, 2, dim=1)
        y = torch.sigmoid(gate) * torch.tanh(filter)
        y = self.output_projection(y)
        residual, skip = torch.chunk(y, 2, dim=1)
        return (x + residual) / sqrt(len(self.residual_layers)) if hasattr(self, 'residual_layers') else (x + residual) / sqrt(2.0), skip

class DiffWave(nn.Module):
    def __init__(self, params):
        super().__init__()
        self.params = params
        self.input_projection = Conv1d(1, params.residual_channels, 1)
        self.diffusion_embedding = DiffusionEmbedding(len(params.noise_schedule))
        if self.params.unconditional:
            self.spectrogram_upsampler = None
            self.label_embedding = None
            self.text_linear = None
            self.condition_fusion = None
        else:
            self.spectrogram_upsampler = SpectrogramUpsampler(params.n_mels)
            self.label_embedding = nn.Embedding(params.num_labels, params.n_mels)
            self.text_linear = Linear(512, params.n_mels)
            # Hierarchical fusion module
            self.condition_fusion = HierarchicalConditionFusion(params.n_mels)

        self.residual_layers = nn.ModuleList([
            ResidualBlock(params.n_mels, params.residual_channels, 2**(i % params.dilation_cycle_length), uncond=params.unconditional)
            for i in range(params.residual_layers)
        ])
        self.skip_projection = Conv1d(params.residual_channels, params.residual_channels, 1)
        self.output_projection = Conv1d(params.residual_channels, 1, 1)
        nn.init.zeros_(self.output_projection.weight)

    def forward(self, audio, diffusion_step, spectrogram=None, label=None, text_emb=None):
        x = audio.unsqueeze(1)  # [B,1,T]
        x = self.input_projection(x)
        x = F.relu(x)
        diffusion_step = self.diffusion_embedding(diffusion_step)

        if self.spectrogram_upsampler is not None:
            upspec = self.spectrogram_upsampler(spectrogram)

            # conditions
            label_emb = None
            text_proj = None

            if label is not None and self.label_embedding is not None:
                label_emb = self.label_embedding(label)
                label_emb = label_emb.unsqueeze(-1)

            if text_emb is not None and self.text_linear is not None:
                text_proj = self.text_linear(text_emb)
                text_proj = text_proj.unsqueeze(-1)

            upspec = self.condition_fusion(upspec, label_emb, text_proj)
        else:
            upspec = None

        skip = None
        for layer in self.residual_layers:
            x, skip_connection = layer(x, diffusion_step, upspec)
            skip = skip_connection if skip is None else skip_connection + skip
        x = skip / sqrt(len(self.residual_layers))
        x = self.skip_projection(x)
        x = F.relu(x)
        x = self.output_projection(x)
        return x

In [ ]:
class Collator:
    def __init__(self, params):
        self.params = params
    def collate(self, minibatch):
        samples_per_frame = self.params.hop_samples
        for i, record in enumerate(minibatch):
            # record: (audio, spectrogram, class_id, text_embedding)
            if not self.params.unconditional:
                if record[1].shape[1] < self.params.crop_mel_frames:
                    minibatch[i] = None
                    continue
                start = random.randint(0, record[1].shape[1] - self.params.crop_mel_frames)
                end = start + self.params.crop_mel_frames
                # cropping spectrogram
                spec_cropped = record[1][:, start:end]
                # cropping audio
                start_audio = start * samples_per_frame
                end_audio = end * samples_per_frame
                audio_cropped = record[0][start_audio:end_audio]
                audio_cropped = np.pad(audio_cropped, (0, (end_audio - start_audio) - len(audio_cropped)), mode='constant')
                minibatch[i] = (audio_cropped, spec_cropped, record[2], record[3])
            else:
                if len(record[0]) < self.params.audio_len:
                    minibatch[i] = None
                    continue
                start = random.randint(0, len(record[0]) - self.params.audio_len)
                end = start + self.params.audio_len
                audio_cropped = record[0][start:end]
                audio_cropped = np.pad(audio_cropped, (0, (end - start) - len(audio_cropped)), mode='constant')
                minibatch[i] = (audio_cropped, record[1], record[2], record[3])
        minibatch = [r for r in minibatch if r is not None]

        audio = np.stack([r[0] for r in minibatch])

        if audio.ndim == 3 and audio.shape[1] == 1:
            audio = np.squeeze(audio, axis=1)
        spectrogram = np.stack([r[1] for r in minibatch])
        labels = [r[2] for r in minibatch]
        text_emb = np.stack([r[3].numpy() for r in minibatch])
        return {
            'audio': torch.from_numpy(audio),
            'spectrogram': torch.from_numpy(spectrogram),
            'class_id': torch.tensor(labels, dtype=torch.long),
            'text_emb': torch.from_numpy(text_emb).float()
        }

In [ ]:
df = df_birds.copy()
df['relative_path'] = df['relative_path'].str.replace(r'/data/\d+/', '/data_new/', regex=True)
df['relative_path'] = df['relative_path'].str.replace('.wav', '_resampled.wav')
df.head()

In [ ]:
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_dataset = SoundDataset(train_df, "text_embeddings.npy")
valid_dataset = SoundDataset(valid_df, "text_embeddings.npy")

collator = Collator(params)
train_dl = DataLoader(train_dataset, batch_size=params.batch_size, shuffle=True,
                      num_workers=0,  collate_fn=collator.collate)
val_dl = DataLoader(valid_dataset, batch_size=params.batch_size, shuffle=False,
                    num_workers=0,  collate_fn=collator.collate)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
vocoder = DiffWave(params).to(device)

beta = np.array(params.noise_schedule)
noise_level = np.cumprod(1 - beta)
vocoder.noise_level = torch.tensor(noise_level.astype(np.float32), device=device)

optimizer = optim.Adam(vocoder.parameters(), lr=params.learning_rate)
epochs = 32

print("Before training:", list(vocoder.parameters())[0].mean().item())
for epoch in range(1, epochs+1):
    vocoder.train()
    total_train_loss = 0.0
    for batch in train_dl:
        audio = batch['audio'].to(device)
        label = batch['class_id'].to(device)
        spec = batch['spectrogram'].to(device)
        text_emb = batch['text_emb'].to(device)
        N, T = audio.shape

        t = torch.randint(0, len(params.noise_schedule), [N], device=device)  # [B]
        noise_scale = vocoder.noise_level[t].unsqueeze(1)
        noise_scale_sqrt = noise_scale ** 0.5
        noise = torch.randn_like(audio)
        noisy_audio = noise_scale_sqrt * audio + ((1.0 - noise_scale) ** 0.5) * noise

        predicted = vocoder(noisy_audio, t, spec, label, text_emb).squeeze(1)
        loss = nn.L1Loss()(noise, predicted)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vocoder.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_dl)
    print(f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}")

    vocoder.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_dl:
            audio = batch['audio'].to(device)
            label = batch['class_id'].to(device)
            spec = batch['spectrogram'].to(device)
            text_emb = batch['text_emb'].to(device)
            N, T = audio.shape
            t = torch.randint(0, len(params.noise_schedule), [N], device=device)
            noise_scale = vocoder.noise_level[t].unsqueeze(1)
            noise_scale_sqrt = noise_scale**0.5
            noise = torch.randn_like(audio)
            noisy_audio = noise_scale_sqrt * audio + ((1.0 - noise_scale)**0.5) * noise
            predicted = vocoder(noisy_audio, t, spec, label, text_emb).squeeze(1)
            val_loss = nn.L1Loss()(noise, predicted)
            total_val_loss += val_loss.item()
    avg_val_loss = total_val_loss / len(val_dl)
    print(f"Epoch {epoch}, Val Loss: {avg_val_loss:.4f}")

print("After training:", list(vocoder.parameters())[0].mean().item())


In [ ]:
save_path = '/content/vocoder_diffwave.pth'
torch.save(vocoder.state_dict(), save_path)

print(f'Model saved to: {save_path}')

## inference

In [ ]:
%%writefile infer.py
import os
import numpy as np
import torch
import torchaudio
from argparse import ArgumentParser
from params import AttrDict, params as base_params
from diffwave import DiffWave
from transformers import T5Tokenizer, T5EncoderModel

models = {}

def perturb_spectrogram(spectrogram, noise_scale=0.05):
    noise = torch.randn_like(spectrogram) * noise_scale
    perturbed_spec = spectrogram + noise
    return perturbed_spec

def predict(spectrogram=None, label=None, text_emb=None, model_dir=None, params=None,
            device=torch.device('cuda'), fast_sampling=False):
    
    # load model.
    if not model_dir in models:
        checkpoint_path = f'{model_dir}/weights.pt' if os.path.exists(f'{model_dir}/weights.pt') else model_dir
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model = DiffWave(AttrDict(base_params)).to(device)
        model.load_state_dict(checkpoint)
        model.eval()
        models[model_dir] = model

    model = models[model_dir]
    model.params.override(params)

    with torch.no_grad():
        training_noise_schedule = np.array(model.params.noise_schedule)
        inference_noise_schedule = (np.array(model.params.inference_noise_schedule)
                                    if fast_sampling else training_noise_schedule)
        talpha = 1 - training_noise_schedule
        talpha_cum = np.cumprod(talpha)
        beta = inference_noise_schedule
        alpha = 1 - beta
        alpha_cum = np.cumprod(alpha)

        T = []
        for s in range(len(inference_noise_schedule)):
            for t in range(len(training_noise_schedule) - 1):
                if talpha_cum[t+1] <= alpha_cum[s] <= talpha_cum[t]:
                    twiddle = (talpha_cum[t]**0.5 - alpha_cum[s]**0.5) / (talpha_cum[t]**0.5 - talpha_cum[t+1]**0.5)
                    T.append(t + twiddle)
                    break
        T = np.array(T, dtype=np.float32)

        if not model.params.unconditional:
            # increase the dimension if spec is 2-dimension
            if len(spectrogram.shape) == 2:
                spectrogram = spectrogram.unsqueeze(0)
            spectrogram = spectrogram.to(device)
            if not isinstance(label, torch.Tensor):
                label = torch.tensor(label, device=device)
            else:
                label = label.to(device)
            audio = torch.randn(spectrogram.shape[0],
                                model.params.hop_samples * spectrogram.shape[-1],
                                device=device)
            if text_emb.ndim == 2 and text_emb.shape[0] == 1:
                text_emb = text_emb.expand(spectrogram.shape[0], -1)
            text_emb = text_emb.to(device)
        else:
            audio = torch.randn(1, params.audio_len, device=device)

        noise_scale = torch.from_numpy(alpha_cum**0.5).float().unsqueeze(1).to(device)

        for n in range(len(alpha) - 1, -1, -1):
            c1 = 1 / (alpha[n]**0.5)
            c2 = beta[n] / ((1 - alpha_cum[n])**0.5)
            audio = c1 * (audio - c2 * model(audio,
                                              torch.tensor([T[n]], device=audio.device),
                                              spectrogram,
                                              label,
                                              text_emb).squeeze(1))
            if n > 0:
                noise = torch.randn_like(audio)
                sigma = (((1.0 - alpha_cum[n-1]) / (1.0 - alpha_cum[n]) * beta[n])**0.5)
                audio += sigma * noise
            audio = torch.clamp(audio, -1.0, 1.0)
    return audio, model.params.sample_rate

def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if args.spectrogram_path:
        spectrogram = torch.from_numpy(np.load(args.spectrogram_path))
        if args.perturb_scale > 0:
            print(f"Perturbing spectrogram with scale: {args.perturb_scale}")
            spectrogram = perturb_spectrogram(spectrogram, noise_scale=args.perturb_scale)
    else:
        spectrogram = None

    if args.label is not None:
        label = int(args.label)
    else:
        label = None

    if args.text is not None:
        tokenizer = T5Tokenizer.from_pretrained("t5-small")
        encoder = T5EncoderModel.from_pretrained("t5-small")
        inputs = tokenizer(args.text, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            text_embedding = encoder(**inputs).last_hidden_state.mean(dim=1)  # shape: [1, hidden_size], 假设 hidden_size=512
    else:
        raise ValueError("Text prompt must be provided for conditional generation.")

    text_embedding = text_embedding.to(device)

    # predict
    audio, sr = predict(spectrogram=spectrogram,
                        label=label,
                        text_emb=text_embedding,
                        model_dir=args.model_dir,
                        fast_sampling=args.fast,
                        params=base_params,
                        device=device)
    torchaudio.save(args.output, audio.cpu(), sample_rate=sr)
    print(f"Output saved as: {args.output}")

if __name__ == '__main__':
    parser = ArgumentParser(description='Text-conditioned music inference with DiffWave')
    parser.add_argument('model_dir', help='Directory or file path to the trained model weights (e.g., weights.pt)')
    parser.add_argument('--spectrogram_path', '-s', help='Path to a spectrogram file (npy) generated by preprocessing')
    parser.add_argument('--output', '-o', default='output.wav', help='Output audio file name')
    parser.add_argument('--fast', '-f', action='store_true', help='Use fast sampling procedure')
    parser.add_argument('--label', '-l', default=None, help='Label (class id) for conditional generation')
    parser.add_argument('--text', '-t', default=None, help='Text prompt for conditional generation')
    parser.add_argument('--perturb_scale', type=float, default=0.0,
                        help='Perturbation scale for adding noise to the spectrogram (set 0 for no perturbation)')
    main(parser.parse_args())


In [ ]:
%%bash
python infer.py /content/vocoder_diffwave.pth \
    --spectrogram_path /data_new/94121_1_resampled.wav.spec.npy \
    --label 19 \
    --text "A robust honking call with steady cadence" \
    --output output.wav